In [1]:
import requests
import re
import time
from pathlib import Path
from typing import List, Dict
from urllib.parse import urljoin
from dataclasses import dataclass
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm import tqdm

In [2]:
@dataclass
class Config:
    """Configuration settings for scraping."""
    max_retries: int = 3
    request_timeout: int = 30
    delay_between_requests: float = 2.0
    max_content_length: int = 10000

# Initialize config
config = Config()

In [3]:
class GoogleSheetsManager:
    """Manages interactions with Google Sheets."""
    
    def __init__(self, credentials_file: str):
        self.credentials_file = credentials_file
        self.service = self._setup_service()
    
    def _setup_service(self):
        """Initialize Google Sheets API service."""
        if not Path(self.credentials_file).exists():
            raise FileNotFoundError(f"Credentials file not found: {self.credentials_file}")
        
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(
            self.credentials_file, scopes=scopes
        )
        return build('sheets', 'v4', credentials=creds)
    
    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """Extract spreadsheet ID from Google Sheets URL."""
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")
    
    def get_urls(self, spreadsheet_id: str, range_name: str = "C:C") -> List[str]:
        """Retrieve URLs from Google Sheet."""
        try:
            result = self.service.spreadsheets().values().get(
                spreadsheetId=spreadsheet_id,
                range=range_name
            ).execute()
            values = result.get('values', [])
            # Skip header row and extract URLs
            urls = [row[0] for row in values[1:] if row and row[0].strip()]
            return urls
        except Exception as e:
            print(f"❌ Error fetching URLs: {e}")
            return []
    
    def update_results(self, spreadsheet_id: str, results: List[Dict]):
        """Update Google Sheet with scraping results, with retry mechanism."""
        if not results:
            return
        
        max_retries = config.max_retries
        for attempt in range(max_retries):
            try:
                # Prepare data
                content_data = [[result['content']] for result in results]
                status_data = [[result['status']] for result in results]
                
                # Update content column (K)
                self.service.spreadsheets().values().update(
                    spreadsheetId=spreadsheet_id,
                    range=f"K2:K{len(content_data) + 1}",
                    valueInputOption='RAW',
                    body={'values': content_data}
                ).execute()
                
                # Update status column (R)
                self.service.spreadsheets().values().update(
                    spreadsheetId=spreadsheet_id,
                    range=f"R2:R{len(status_data) + 1}",
                    valueInputOption='RAW',
                    body={'values': status_data}
                ).execute()
                
                print(f"✅ Updated {len(results)} rows in Google Sheet")
                return
            except HttpError as e:
                print(f"❌ HTTP Error on attempt {attempt + 1}: {e}")
                if e.resp.status == 429:
                    print("Rate limit exceeded.")
            except Exception as e:
                print(f"❌ Error on attempt {attempt + 1}: {e}")
            
            if attempt < max_retries - 1:
                wait_time = 5 * (attempt + 1)  # Exponential backoff
                print(f"⏳ Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("❌ All retries failed. Could not update sheet.")

In [4]:
class WebScraper:
    """Web scraper for about and mission pages."""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        })
        self.session.verify = True
    
    def _normalize_url(self, url: str) -> str:
        """Add https if missing."""
        if not url.startswith(('http://', 'https://')):
            url = 'https://' + url
        return url
    
    def _get_about_urls(self, base_url: str) -> List[str]:
        """Generate potential about page URLs."""
        base_url = self._normalize_url(base_url)
        about_paths = [
            '/about', '/about-us', '/about/', '/about.html',
            '/company', '/company/', '/company-info',
            '/who-we-are', '/who-we-are/', '/our-story',
            '/our-company', '/overview', '/profile',
            '/team', '/leadership', '/history'
        ]
        return [urljoin(base_url, path) for path in about_paths]
    
    def _get_mission_urls(self, base_url: str) -> List[str]:
        """Generate potential mission page URLs."""
        base_url = self._normalize_url(base_url)
        mission_paths = [
            '/mission', '/mission/', '/mission-vision',
            '/our-mission', '/vision', '/values',
            '/purpose', '/why-we-exist', '/what-we-do',
            '/mission-statement', '/vision-mission',
            '/core-values', '/beliefs', '/philosophy'
        ]
        return [urljoin(base_url, path) for path in mission_paths]
    
    def _extract_content(self, soup: BeautifulSoup) -> str:
        """Extract meaningful content from HTML."""
        for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'form', 'button']):
            element.decompose()
        
        selectors = [
            '.about-content, .about-section, #about, .about-wrapper',
            '.mission-content, .mission-section, #mission, .mission-wrapper',
            '.company-info, .company-overview, .company-story',
            '.our-story, .our-mission, .our-vision, .our-values',
            'main, .main-content, .content, .page-content',
            'article, .article-content, .post-content',
            '.entry-content, .text-content, .body-content',
            '.container .content, .wrapper .content',
            'section[class*="about"], section[class*="mission"]',
            'div[class*="about"], div[class*="mission"]',
            'section[id*="about"], section[id*="mission"]',
            'div[id*="about"], div[id*="mission"]',
            '.row .col, .grid .item, .flex .content',
            '.hero-content, .intro-content, .description'
        ]
        
        for selector in selectors:
            try:
                elements = soup.select(selector)
                if elements:
                    content_parts = [el.get_text(strip=True) for el in elements if el.get_text(strip=True) and len(el.get_text(strip=True)) > 50]
                    if content_parts:
                        content = ' '.join(content_parts)
                        if len(content) > 100:
                            return content[:config.max_content_length]
            except Exception:
                continue
        
        content_elements = soup.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'div'],
                                        class_=lambda x: x and any(keyword in x.lower() for keyword in ['about', 'mission', 'story', 'company', 'vision']))
        if content_elements:
            content = ' '.join([el.get_text(strip=True) for el in content_elements if el.get_text(strip=True) and len(el.get_text(strip=True)) > 20])
            if content:
                return content[:config.max_content_length]
        
        paragraphs = soup.find_all('p')
        if paragraphs:
            meaningful_paragraphs = [p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True) and len(p.get_text(strip=True)) > 30]
            if meaningful_paragraphs:
                content = ' '.join(meaningful_paragraphs)
                return content[:config.max_content_length] if content else "No content found"
        
        return "No content found"
    
    def _try_urls(self, urls: List[str], page_type: str) -> Dict[str, str]:
        """Attempt to scrape a list of URLs."""
        for url in urls:
            try:
                response = self.session.get(url, timeout=config.request_timeout, allow_redirects=True)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    content = self._extract_content(soup)
                    if content and content != "No content found" and len(content.strip()) > 100:
                        return {'content': content, 'status': f'success - {page_type} page found', 'found_url': url}
            except Exception:
                continue
            time.sleep(0.3)
        return None
    
    def scrape_about_page(self, base_url: str) -> Dict[str, str]:
        """Scrape about page, with fallbacks to mission page and homepage."""
        result = {'url': base_url, 'content': '', 'status': 'failed', 'found_url': ''}
        
        about_urls = self._get_about_urls(base_url)
        about_result = self._try_urls(about_urls, 'about')
        if about_result:
            result.update(about_result)
            return result
        
        mission_urls = self._get_mission_urls(base_url)
        mission_result = self._try_urls(mission_urls, 'mission')
        if mission_result:
            result.update(mission_result)
            return result
        
        try:
            homepage_url = self._normalize_url(base_url)
            response = self.session.get(homepage_url, timeout=config.request_timeout, allow_redirects=True)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                content = self._extract_content(soup)
                if content and content != "No content found" and len(content.strip()) > 100:
                    result.update({'content': content, 'status': 'success - homepage content', 'found_url': homepage_url})
                    return result
        except Exception:
            pass
        
        result['status'] = 'failed - no about, mission, or meaningful homepage content found'
        return result
    
    def scrape_multiple(self, urls: List[str]) -> List[Dict[str, str]]:
        """Scrape multiple URLs with progress tracking."""
        results = []
        for url in tqdm(urls, desc="Scraping websites"):
            result = self.scrape_about_page(url)
            results.append(result)
            time.sleep(config.delay_between_requests)
            status = "✅" if result['status'].startswith('success') else "❌"
            found_info = f" (found: {result.get('found_url', '').split('/')[-1] or 'homepage'})" if status == "✅" else ""
            print(f"{status} {url}{found_info} - {result['status']}")
        return results

In [5]:
def scrape_from_google_sheet(credentials_file: str, sheet_url: str, batch_size: int = 100):
    """Main function to scrape websites from Google Sheet."""
    try:
        print("🚀 Initializing scraper...")
        sheets_manager = GoogleSheetsManager(credentials_file)
        scraper = WebScraper()
        
        spreadsheet_id = sheets_manager.extract_spreadsheet_id(sheet_url)
        print(f"📊 Spreadsheet ID: {spreadsheet_id}")
        
        print("📋 Fetching URLs from Google Sheet...")
        urls = sheets_manager.get_urls(spreadsheet_id)
        
        if not urls:
            print("❌ No URLs found in the sheet")
            return None
        
        print(f"📝 Found {len(urls)} URLs to scrape")
        
        if len(urls) > 100:
            confirm = input(f"WARNING: Scraping {len(urls)} URLs may take over {len(urls) * 10 // 3600} hours. Proceed? (y/n): ")
            if confirm.lower() != 'y':
                print("Scraping cancelled.")
                return None
        
        results = []
        for i in tqdm(range(0, len(urls), batch_size), desc="Processing batches"):
            batch_urls = urls[i:i + batch_size]
            batch_results = scraper.scrape_multiple(batch_urls)
            results.extend(batch_results)
            print(f"📤 Updating Google Sheet with batch {(i // batch_size) + 1}...")
            sheets_manager.update_results(spreadsheet_id, results)
        
        successful = sum(1 for r in results if r['status'].startswith('success'))
        failed = len(results) - successful
        success_rate = (successful / len(results) * 100) if results else 0
        
        print("\n" + "="*50)
        print("📊 SCRAPING SUMMARY")
        print("="*50)
        print(f"Total URLs: {len(results)}")
        print(f"Successful: {successful}")
        print(f"Failed: {failed}")
        print(f"Success Rate: {success_rate:.1f}%")
        
        return {'total_urls': len(results), 'successful': successful, 'failed': failed, 'success_rate': f"{success_rate:.1f}%"}
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

In [6]:

CREDENTIALS_FILE = ""
GOOGLE_SHEET_URL = "" 

# - Column C: Website URLs (starting from row 2)
# - Column K: Will be filled with scraped content
# - Column R: Will be filled with status information

if __name__ == "__main__":
    results = scrape_from_google_sheet(CREDENTIALS_FILE, GOOGLE_SHEET_URL)

🚀 Initializing scraper...
📊 Spreadsheet ID: 152clSgMA-9oW1xFotbToo0ci8DjgaWQDxYNL6Smpn9M
📋 Fetching URLs from Google Sheet...
📝 Found 36 URLs to scrape


aping websites:   3%|█▊                                                              | 1/36 [00:04<02:41,  4.62s/it]

✅ https://www.marcusmillichap.com/ (found: about-us) - success - about page found



aping websites:   6%|███▌                                                            | 2/36 [00:07<02:06,  3.72s/it]

✅ https://www.axiscapital.com/ (found: about) - success - about page found


...

✅ https://www.heritagehomeinvestments.com/ (found: our-company) - success - about page found



Scraping websites: 100%|███████████████████████████████████████████████████████████████| 36/36 [05:47<00:00,  9.64s/it]

✅ https://simplecfo.com/ (found: about) - success - about page found
📤 Updating Google Sheet with batch 1...
❌ Error on attempt 1: EOF occurred in violation of protocol (_ssl.c:2406)
⏳ Retrying in 5 seconds...



Processing batches: 100%|███████████████████████████████████████████████████████████████| 1/1 [05:54<00:00, 354.09s/it]

✅ Updated 36 rows in Google Sheet

📊 SCRAPING SUMMARY
Total URLs: 36
Successful: 34
Failed: 2
Success Rate: 94.4%


In [7]:

results = scrape_from_google_sheet(CREDENTIALS_FILE, GOOGLE_SHEET_URL)


🚀 Initializing scraper...
📊 Spreadsheet ID: 152clSgMA-9oW1xFotbToo0ci8DjgaWQDxYNL6Smpn9M
📋 Fetching URLs from Google Sheet...
📝 Found 36 URLs to scrape


aping websites:   3%|█▊                                                              | 1/36 [00:05<03:14,  5.55s/it]

✅ https://www.marcusmillichap.com/ (found: about-us) - success - about page found



aping websites:   6%|███▌                                                            | 2/36 [00:08<02:16,  4.01s/it]

✅ https://www.axiscapital.com/ (found: about) - success - about page found


...

✅ https://www.heritagehomeinvestments.com/ (found: our-company) - success - about page found



Scraping websites: 100%|███████████████████████████████████████████████████████████████| 36/36 [05:47<00:00,  9.65s/it]

✅ https://simplecfo.com/ (found: about) - success - about page found
📤 Updating Google Sheet with batch 1...
❌ Error on attempt 1: EOF occurred in violation of protocol (_ssl.c:2406)
⏳ Retrying in 5 seconds...



Processing batches: 100%|███████████████████████████████████████████████████████████████| 1/1 [05:54<00:00, 354.83s/it]

✅ Updated 36 rows in Google Sheet

📊 SCRAPING SUMMARY
Total URLs: 36
Successful: 34
Failed: 2
Success Rate: 94.4%
